# FPGA Security Evaluation — Reproducibility Notebook

**Dissertation:** *Machine Learning on the Edge: Performance and Security Evaluation of Artificial Neural Network Implementations in Embedded Systems*  
**Results section:** 4.7.2.1 — FPGA Deployment  
**Target:** ZedBoard Revision E; Zynq-7020 (`XC7Z020CLG484-1`)  
**Development tools:** Vivado 2018.1; Xilinx SDK 2018.1  
**Review status:** Qualitative configuration and documentation review; **not** a penetration test.

This notebook documents how the FPGA security evaluation was performed and how findings were checked against the implemented configuration. It separates **observed evidence**, **investigator-reported operating conditions**, **vendor advisories**, and **unverified questions**. It does not test vulnerabilities, probe networks, modify hardware, or make claims about the security of systems beyond the experiment.

**Repository placement:** Store this notebook in your `FPGA Security` branch alongside the screenshots `write_bitstream_log.png` and `bitstream_settings.png`. The `Manuals and Specifications` branch contains the documentation/reference list. Since Git branches are separate trees, **relative filesystem links do not cross branches**; use the official source links supplied below and add the actual GitHub branch URL once known.

## 1. Evidence inventory and provenance

| ID | Evidence | Type | What it supports | What it does **not** prove |
|---|---|---|---|---|
| E01 | `write_bitstream_log.png` | Vivado screenshot | Command `write_bitstream -force design_1_wrapper.bit` and successful bitstream generation | Encryption or authentication status |
| E02 | `bitstream_settings.png` | Vivado screenshot | Visible Bitstream settings did not show enabled security options | All design/boot security properties were disabled |
| E03 | Vivado design-property search (investigator report) | Reported configuration check | No properties matching `BITSTREAM`, `ENCRYPTION`, or `SECURITY` were found in the review | Exhaustive proof that encryption was absent |
| E04 | Vivado hardware hierarchy (reviewed separately) | HDL hierarchy | Neuron-level `Weight_Memory` instances exist | Runtime write access by an external attacker |
| E05 | `include.v`, `Weight_Memory.v`, `neuron.v` (source in implementation branch) | Source review | `pretrained` selects `$readmemb` initialization and excludes alternate normal write path | Cryptographic protection of weights; no other modification routes |
| E06 | Physical and remote access notes | Investigator report | Private office; JTAG/UART attached except power and standalone proof; Windows PIN; remote-control permission from a separate Quest headset | Unauthorized access or exploit success |
| E07 | SDK / boot workflow notes | Investigator report | Direct JTAG workflow, SDK separately launched for cloud ARM application; no `fsbl.elf`/`BOOT.BIN` identified in project | An FSBL never executed in any environment |
| E08 | AMD UG585 and relevant advisories | Vendor documentation | Security capabilities and advisory prerequisites | That any supported protection was enabled in this project |
| E09 | Vivado 2018.1 IP change log review | Vendor documentation + investigator review | ILA 6.2 XDC warning update; other reviewed IP components listed as unchanged | That all IP is vulnerability-free |
| E10 | OV7670 paper by Liu et al. (2024) | Published comparison | Example of camera->FPGA->BRAM->encryption->network pipeline | A known OV7670 vulnerability or encryption in this experiment |

**Source-code reproducibility note:** The source files in E05 belong in the corresponding implementation branch. Add links to their immutable Git commit/paths in the table or branch README; this notebook does not assume that those files are present here.

In [ ]:
from pathlib import Path
import hashlib
import csv
from datetime import date
from IPython.display import display, Image, Markdown

ROOT = Path.cwd()
SCREENSHOTS = ["write_bitstream_log.png", "bitstream_settings.png"]

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for fname in SCREENSHOTS:
    path = ROOT / fname
    if path.is_file():
        print(f"FOUND {fname} | bytes={path.stat().st_size} | SHA256={sha256(path)}")
    else:
        print(f"MISSING {fname} — place it beside this notebook in the FPGA Security branch")

## 2. Screenshot review — bitstream generation

1. Open `write_bitstream_log.png`; confirm the `write_bitstream` invocation and `Bitgen Completed Successfully` message.
2. Record the actual generated filename (`design_1_wrapper.bit` in the screenshot).
3. Confirm the log's DRC messages concern DSP output pipelining. These warnings are **not**, by themselves, cybersecurity advisories.
4. Open `bitstream_settings.png` and record visible options. Note that the screenshot states additional settings may become available after opening an implemented design.
5. In Vivado, optionally run `report_property [current_design]` **after opening the implemented design**. Capture the output and the exact search method if a fuller audit is needed.

**Interpretation rule:** Lack of a visible property or explicit encryption flag permits the statement “no encryption/authentication setting was identified in the reviewed evidence,” **not** “encryption was conclusively disabled.”

In [ ]:
for fname in SCREENSHOTS:
    path = ROOT / fname
    if path.is_file():
        display(Markdown(f"**{fname}**"))
        display(Image(filename=str(path)))

### Relevant non-destructive Vivado verification commands

Run these in the Vivado Tcl Console **only when the original implemented design is open**. Save the output in the `FPGA Security` branch if available. They are provided for reproducibility and were **not all independently executed by this notebook**.

```tcl
# Configuration and build evidence
report_property [current_design]
report_property [current_project]
get_files -all *.bit

# Capture version and the configured project files
version
get_files -all
```

The original observed command was:

```tcl
write_bitstream -force design_1_wrapper.bit
```

The generated design may have received security options from upstream project constraints/Tcl rather than the command line alone. A complete conclusive review would inspect the actual project settings, script history, and selected bitstream properties. Do **not** rerun bitstream generation just to recreate the screenshots unless intentionally replicating the build.

## 3. Actual deployment and access conditions

The following conditions are investigator-reported, not inferred from the screenshots:

- **Programming:** JTAG through Vivado 2018.1 for both local and cloud-assisted FPGA designs.
- **Cloud ARM application:** Xilinx SDK 2018.1 launched separately.
- **Location:** Private office; no documented public access.
- **JTAG and UART:** Connected for most tests; disconnected for power measurements and standalone demonstration of both local and cloud-assisted classification.
- **Workstation:** Windows sign-in protected by a PIN. Vivado and SDK were accessible to users with an active Windows session.
- **Remote workstation access:** The *built-in Remote Desktop application* on a **different** Meta Quest 3S headset (not the experimental headset) had been allowed to remotely control an active Windows session. Vendor authentication and transport protections for that feature were **not independently verified**.
- **Boot files:** The investigator did not locate `fsbl.elf` or `BOOT.BIN` in the reviewed project. This supports, but does **not** conclusively prove, absence of an FSBL execution step.

**Threat scenario:** Someone with physical access to the connected debug/programming environment—or effective remote control of an already-active Windows session—could potentially change project artifacts or interact with connected devices. This was **not tested**. Disconnected cable configurations demonstrated operational independence, not total resistance to tampering.

## 4. Local NN parameters: source-code review

The reviewed hardware-generation materials (`mnistZyNet2.py`, `Weight_Memory.v`, `neuron.v`, `include.v`) are stored with implementation files, not necessarily in this security branch.

**Observed source features:**

- Python model preparation called `genWeightArray(...)` and `genBiasArray(...)`, then compiled with `pretrained="Yes"`.
- `include.v` contained `` `define pretrained ``.
- `Weight_Memory.v` used `$readmemb(weightFile, mem)` under `` `ifdef pretrained ``; its alternate `always @(posedge clk)` write branch was under `` `else ``.
- `neuron.v` initialized `biasReg` via `$readmemb(biasFile, biasReg)` when pretrained mode was selected; the alternative external bias update branch was conditional on the opposite compile path.

**Supported result:** In the reviewed source configuration, the ordinary neuron-level runtime write paths for pretrained weights and biases were excluded by conditional compilation. This narrows the parameter modification surface *through those module paths*. It does not authenticate the `.mif`/initialization files, generated HDL, or programmed bitstream, and it does not establish immunity to physical reconfiguration.

**Important scope note:** The FPGA's cloud-assisted design did not run the local zyNet classifier: it sent prepared images to Google Cloud Run using PL/PS shared memory. Detailed Ethernet/TLS and cloud-server findings belong in Chapter IV Section 4.7.3, not in this local FPGA hardware review.

## 5. Vendor and vulnerability references

The external URLs below correspond to the reference list documented in the **`Manuals and Specifications` branch README**. These are direct vendor/database URLs so they remain useful before you insert the GitHub repository and branch name. GitHub branch links use `/blob/<branch-name>/README.md`, where spaces must be URL-encoded.

**Core documentation**

- [AMD Zynq-7000 SoC Technical Reference Manual (UG585)](https://docs.amd.com/r/en-US/ug585-zynq-7000-SoC-TRM/Introduction?tocId=Hf6C7Oo5ABvv2hkWRoiihQ) — device family, security capabilities, JTAG and boot behavior.
- [AMD Design Advisory 73541 — Defeating Bitstream Encryption](https://docs.amd.com/r/en-US/73541/Solution) — attack and countermeasures for encrypted 7-Series configurations.
- [AMD Answer Record 70394 — Meltdown and Spectre](https://adaptivesupport.amd.com/s/article/70394?language=en_US) — Zynq-7000 Cortex-A9 applicability.
- [AMD Answer Record 71292 — FSBL partition-header security](https://adaptivesupport.amd.com/s/article/71292?language=en_US) — Zynq-7000 2018.2 and prior FSBL issue.
- [AMD Answer Record 76974 — FSBL authentication attack](https://adaptivesupport.amd.com/s/article/76974?language=en_US) — FSBL authentication issue and prerequisites.
- [AMD Answer Record 70699 — Vivado 2018.1 IP change log](https://adaptivesupport.amd.com/s/article/70699?language=en_US) — release-note review, not inherently a vulnerability notice.
- [OpenCVE Zynq-7000 index](https://app.opencve.io/cve/?product=zynq-7000&vendor=xilinx) — discovery index; prefer individual primary advisories for claims.
- [NVD CVE-2021-44850](https://nvd.nist.gov/vuln/detail/CVE-2021-44850) — SD-card boot issue.
- [NVD CVE-2022-23822](https://nvd.nist.gov/vuln/detail/CVE-2022-23822) — FSBL authentication vulnerability.
- [NVD CVE-2021-27208](https://nvd.nist.gov/vuln/detail/CVE-2021-27208) — NAND boot issue.
- [AMD Answer Record 66071](https://adaptivesupport.amd.com/s/article/66071?language=en_US) — **Zynq UltraScale+**; different device family and screened out.
- [NVD CVE-2025-52541](https://nvd.nist.gov/vuln/detail/CVE-2025-52541) — Vivado issue reported for newer versions; 2018.1 applicability **unverified**, not a confirmed finding.
- [AMD community discussion on 7-Series encryption](https://adaptivesupport.amd.com/s/question/0D52E00006hpX05SAE/broken-7series-encryption-and-countermeasures-of-zynq7000?language=en_US) — supplementary; primary AMD advisory preferred.

**Camera-related published example:** [Liu et al. (2024), *Real-Time Secure Video Streaming System Based on FPGA and CUDA Technology*](https://doi.org/10.1145/3711618.3711642). It used an OV7670 camera, BRAM, and FPGA-based encryption on **different hardware**; it is not evidence of a vulnerability in the OV7670 or evidence that this dissertation used that encryption method.

> **Repository cross-branch link (set after upload):** replace `OWNER`, `REPOSITORY`, and `MANUALS_BRANCH` in the next cell to produce a link to the Manuals and Specifications branch README. These placeholders are intentional: the GitHub repository URL and exact branch slug were not provided.

In [ ]:
from urllib.parse import quote
OWNER = "OWNER"              # TODO: set GitHub account / organization
REPOSITORY = "REPOSITORY"    # TODO: set exact repository name
MANUALS_BRANCH = "MANUALS_BRANCH"  # TODO: exact Git branch name; e.g. Manuals and Specifications
if any(v in ("OWNER", "REPOSITORY", "MANUALS_BRANCH") for v in (OWNER, REPOSITORY, MANUALS_BRANCH)):
    print("Configure GitHub owner, repository, and Manuals branch slug to generate the cross-branch link.")
else:
    readme_url = (f"https://github.com/{quote(OWNER)}/{quote(REPOSITORY)}"
                  f"/blob/{quote(MANUALS_BRANCH, safe='')}/README.md")
    display(Markdown(f"[Manuals and Specifications branch README]({readme_url})"))

## 6. Reproducible applicability screen

Classify each notice using the same decision rules:

1. **Device family:** Does the advisory cover Zynq-7000 / Cortex-A9 / Vivado 2018.1, rather than a different family or version?
2. **Affected feature and prerequisite:** Does the attack need SD-card/NAND boot, authenticated FSBL, encryption, speculative execution, or a Windows installer condition?
3. **Evidence of feature use:** Is the required feature present in the actual programming/run configuration, based on the screenshots, source, project files, or investigator notes?
4. **Result:** Report `architecture relevant`, `prerequisites not established`, `outside platform/version`, `documentation only`, or `version applicability unverified`. **Do not label `not vulnerable` solely because a required artifact was not found.**
5. **Testing:** No attack was performed; relevance describes a documented prerequisite comparison, not an exploit result.

The entries below capture the documented review. Update notes only when new vendor or project evidence is available.

In [ ]:
records = [
    dict(id="A01", source="AMD UG585", component="Zynq-7000 security features", prerequisite="Secure boot, bitstream protection must be configured", evidence="JTAG workflow; no matching security settings identified in reviewed properties", finding="Capabilities documented; enabled status unverified", class_name="configuration not established"),
    dict(id="A02", source="AMD 73541", component="7-Series encrypted bitstream", prerequisite="Target uses relevant encrypted-bitstream scheme", evidence="No encryption setting identified during Vivado review", finding="Encrypted-bitstream attack prerequisites not established", class_name="prerequisites not established"),
    dict(id="A03", source="AMD 70394", component="Zynq-7020 Cortex-A9", prerequisite="Attacker can exercise a relevant speculative-execution scenario", evidence="AMD identifies Spectre V1 CVE-2017-5753 and V2 CVE-2017-5715 susceptibility; ARM PS used for cloud client", finding="Architectural susceptibility; actual exploitability not tested", class_name="architecture relevant"),
    dict(id="A04", source="AMD 71292", component="Zynq-7000 FSBL (2018.2 and earlier)", prerequisite="Affected FSBL authentication/partition-header workflow", evidence="JTAG + separately launched SDK; no fsbl.elf/BOOT.BIN found in project", finding="Affected boot process not established", class_name="prerequisites not established"),
    dict(id="A05", source="AMD 76974 / CVE-2022-23822", component="Zynq-7000 FSBL", prerequisite="Affected authenticated FSBL environment and attack conditions", evidence="No authenticated FSBL boot image identified in project", finding="FSBL attack prerequisites not established", class_name="prerequisites not established"),
    dict(id="A06", source="NVD CVE-2021-44850", component="Zynq-7000 SD boot", prerequisite="Manipulated SD boot image", evidence="Documented JTAG workflow; no SD boot used for tested design", finding="SD boot path not in documented workflow", class_name="prerequisites not established"),
    dict(id="A07", source="NVD CVE-2021-27208", component="Zynq-7000 NAND boot", prerequisite="Affected NAND boot path", evidence="JTAG workflow; NAND boot not documented", finding="NAND boot path not in documented workflow", class_name="prerequisites not established"),
    dict(id="A08", source="AMD 66071", component="Zynq UltraScale+ MPSoC", prerequisite="UltraScale+ device family", evidence="Device is Zynq-7020 Zynq-7000", finding="Different device family", class_name="outside platform/version"),
    dict(id="A09", source="AMD 70699", component="Vivado 2018.1 IP", prerequisite="Security-relevant IP update affecting installed component", evidence="ILA 6.2 XDC warning update; other checked IP listed unchanged", finding="No security issue identified in reviewed change entries", class_name="documentation only"),
    dict(id="A10", source="NVD CVE-2025-52541", component="Windows Vivado", prerequisite="Affected Vivado release/version and DLL search-path conditions", evidence="Published notice discussed later release; 2018.1 applicability not shown", finding="Not reported as confirmed 2018.1 vulnerability", class_name="version applicability unverified"),
]

for r in records:
    print(f"{r['id']} | {r['source']} | {r['class_name']} | {r['finding']}")

## 7. Findings ledger for Chapter IV

Use these labels consistently:

- **Observed configuration**: supported directly by screenshots or inspected HDL.
- **Reported configuration**: supplied by the investigator, not independently recovered from a project artifact.
- **Published architectural vulnerability**: applies to a processor/device family, but exploitability in this application remains untested.
- **Potential threat**: feasible-looking scenario requiring assumptions or access not demonstrated.
- **Out of scope / prerequisites not established**: documentary applicability screen, not proof of invulnerability.

**No numeric security score is calculated.** Qualitative likelihood/impact ratings require explicit reproducible criteria and supported exposure evidence. The ledger below therefore uses descriptive security implications only.

In [ ]:
findings = [
    dict(id="F01", topic="JTAG / bitstream", evidence="E01 E02 E03 E06", type="observed + reported configuration", finding="JTAG programming; no encryption/authentication options identified in reviewed settings", consequence="Potential unauthorized reconfiguration with programming access", limits="Cannot conclusively verify all boot and bitstream protection settings"),
    dict(id="F02", topic="Physical/debug access", evidence="E06", type="reported configuration", finding="Private office; JTAG/UART attached during most tests; disconnected for power and standalone demonstrations", consequence="Connection could permit interaction if physical/workstation access obtained", limits="No unauthorized use tested"),
    dict(id="F03", topic="Remote workstation control", evidence="E06", type="reported configuration", finding="Separate Quest headset permitted remote control of active Windows session; workstation PIN-protected", consequence="Potential access to engineering tools and connected JTAG device", limits="Remote desktop authentication/protocol not independently assessed"),
    dict(id="F04", topic="NN weights and biases", evidence="E04 E05", type="source review", finding="Pretrained readmemb path selected; alternate weight and bias write logic conditionally excluded", consequence="Normal module parameter-write surface reduced; source and bitstream integrity remain", limits="No full attack path or extraction assessment"),
    dict(id="F05", topic="Cortex-A9 Spectre", evidence="E08", type="published architectural vulnerability", finding="AMD states susceptibility to Spectre V1/V2", consequence="Potential disclosure under applicable attack conditions, relevant to cloud-assisted PS", limits="Local PL inference independent of PS; no Spectre test"),
    dict(id="F06", topic="Boot and encryption advisories", evidence="E01 E02 E07 E08", type="prerequisite comparison", finding="SD/NAND, encrypted-bitstream, authenticated FSBL prerequisites not established in the documented workflow", consequence="Not reported as demonstrated exploit paths", limits="Absence of files alone cannot prove no FSBL ever ran"),
    dict(id="F07", topic="Vivado IP updates", evidence="E09", type="release-note review", finding="ILA XDC warning update; no security change identified among checked IP entries", consequence="No confirmed IP advisory from this check", limits="Release notes cannot prove absence of vulnerabilities"),
    dict(id="F08", topic="OV7670 interface", evidence="E10 + dissertation design description", type="architecture + published comparison", finding="Direct camera input to PL; no camera-specific vulnerability identified from reviewed article", consequence="Potential physical input manipulation or acquisition-logic integrity concerns", limits="No camera attack; comparison paper used different FPGA and encryption"),
]
for f in findings:
    print(f"{f['id']}: {f['topic']} — {f['finding']}")

## 8. Export reproducibility records

Run the next cell to export machine-readable screening/ledger files into an `outputs/` folder in your checked-out security branch. Review the exported CSV files before committing. The notebook does **not** download vendor pages or claim to have verified their current availability.

In [ ]:
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

def export_csv(name, rows):
    dst = OUT / name
    with dst.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print(f"Saved {dst.relative_to(ROOT)}")

export_csv("fpga_advisory_applicability.csv", records)
export_csv("fpga_security_findings.csv", findings)

## 9. Chapter IV reporting protocol

For **Section 4.7.2.1 FPGA Deployment**, report in this order:

1. Direct-JTAG programming, standard bitstream generation, and protection settings not verified as enabled (E01–E03; UG585).
2. Physical/debug and remote workstation access conditions (E06), including that the non-experimental Quest headset controlled an active workstation session.
3. Local PL weight and bias initialization in pretrained mode, and limitations of the module-level write paths (E04–E05).
4. Device-family advisories screened by required boot/encryption features (E07–E08).
5. Cortex-A9 Spectre architectural susceptibility, noting that the local PL classifier did not use PS for inference (AMD 70394).
6. Vivado 2018.1 IP release-note observations, without interpreting unchanged entries as proof of software security (E09).
7. OV7670 input interface as a potential physical/input-integrity issue, supported by the *distinct* camera-encryption example from Liu et al. (E10).

**Defer to Section 4.7.3:** FPGA Ethernet transport, lwIP, Mbed TLS, server-certificate verification disabled, deterministic development entropy, Cloud Run service and request access, and the PL/PS BRAM mailbox in its role in cloud-assisted transmission.

**Do not claim:** successful attacks; measured crypto or authentication overhead; all vulnerabilities absent; conclusively disabled bitstream encryption; confirmed remote compromise; or verified vendor mitigations not used in the build.

### Reproduction record to complete

- Review performed on: `2026-11-04` (screen captures and initial review reported that day).
- Original Vivado project commit: **TO FILL**.
- HDL implementation branch/commit: **TO FILL**.
- Manuals and Specifications branch commit: **TO FILL**.
- Investigator and reviewer initials: **TO FILL**.
- Date vendor advisory text last rechecked: **TO FILL**.
- Any screenshot `SHA256` values: generated in Section 1 after files are placed with the notebook.

**Interpretation:** This notebook reproduces the **evaluation procedure and evidence inventory**, not the controlled experimental conditions necessary to validate exploitability.